# 00 — Data, HTML and OCR Audit

**Question.** What information is present in the corpus, which HTML structures must be preserved, and how much useful image context is available?

**Success criteria.** Produce a validated parsed corpus, compact quality tables, and reusable artifacts for all later notebooks.


## Setup and reproducibility


In [ ]:
from pathlib import Path
import os, sys, json, time, copy, itertools, importlib.util, subprocess

def locate_project(start=Path.cwd()):
    for parent in [start.resolve(), *start.resolve().parents]:
        if (parent / "pyproject.toml").exists() and (parent / "src/avito_retriever").exists():
            return parent
    candidate = Path("/content/avito-retriever")
    if candidate.exists():
        return candidate
    raise FileNotFoundError("Clone/open avito-retriever or change PROJECT_ROOT in this cell")

PROJECT_ROOT = locate_project()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

def module_available(name):
    try:
        return importlib.util.find_spec(name) is not None
    except ModuleNotFoundError:
        return False

IN_COLAB = module_available("google.colab")
if IN_COLAB and os.environ.get("AVITO_AUTO_INSTALL", "1") != "0":
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-e",
        f"{PROJECT_ROOT}[lexical,neural,dev]",
    ])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from avito_retriever.experiments.notebook import resolve_data_dir, result_dir, highlight_best, save_json, save_yaml

DATA_DIR = resolve_data_dir(PROJECT_ROOT)
SEED = 42
np.random.seed(SEED)
print("Project:", PROJECT_ROOT)
print("Data:", DATA_DIR)


In [ ]:
from avito_retriever.data.io import load_articles, load_calibration, load_test
from avito_retriever.preprocessing.html import parse_articles, FIELD_COLUMNS
from avito_retriever.preprocessing.images import extract_image_manifest

OUT = result_dir(PROJECT_ROOT, "00_data_html_ocr_audit")
articles = load_articles(DATA_DIR)
calibration = load_calibration(DATA_DIR)
test = load_test(DATA_DIR)


## Dataset contract


In [ ]:
summary = pd.DataFrame([
    {"dataset": "articles", "rows": len(articles), "columns": ", ".join(articles.columns), "missing": int(articles.isna().sum().sum())},
    {"dataset": "calibration", "rows": len(calibration), "columns": ", ".join(calibration.columns), "missing": int(calibration.isna().sum().sum())},
    {"dataset": "test", "rows": len(test), "columns": ", ".join(test.columns), "missing": int(test.isna().sum().sum())},
])
display(summary)
assert articles.article_id.is_unique and calibration.query_id.is_unique and test.query_id.is_unique


## HTML structure inventory


In [ ]:
from bs4 import BeautifulSoup
from collections import Counter

tags, classes = Counter(), Counter()
for html in articles.body:
    soup = BeautifulSoup(str(html), "lxml")
    tags.update(tag.name for tag in soup.find_all(True))
    classes.update(cls for tag in soup.find_all(True) for cls in tag.get("class", []))

html_inventory = pd.DataFrame(tags.most_common(25), columns=["tag", "count"])
class_inventory = pd.DataFrame(classes.most_common(20), columns=["css_class", "count"])
display(html_inventory.head(15), class_inventory.head(15))


## Parse articles into retrieval fields


In [ ]:
parsed_path = PROJECT_ROOT / "artifacts/cache/parsed_articles.parquet"
if parsed_path.exists():
    parsed = pd.read_parquet(parsed_path)
else:
    parsed = parse_articles(articles)
    parsed_path.parent.mkdir(parents=True, exist_ok=True)
    parsed.to_parquet(parsed_path, index=False)

field_stats = pd.DataFrame([
    {"field": field, "non_empty_articles": int(parsed[field].str.len().gt(0).sum()),
     "median_chars": float(parsed[field].str.len().median()), "p95_chars": float(parsed[field].str.len().quantile(.95))}
    for field in FIELD_COLUMNS
])
display(field_stats)


## Image and OCR opportunity


In [ ]:
image_manifest = extract_image_manifest(articles)
image_summary = pd.DataFrame([{
    "images": len(image_manifest),
    "articles_with_images": image_manifest.article_id.nunique(),
    "unique_urls": image_manifest.src.nunique(),
    "non_empty_alt": int(image_manifest.alt.str.len().gt(0).sum()),
    "unique_alt": image_manifest.alt.nunique(),
}])
display(image_summary)
display(image_manifest[["article_id", "alt", "src"]].head(12))


## Query and label profile


In [ ]:
label_counts = calibration.ground_truth.str.split().str.len()
label_profile = pd.DataFrame([{
    "queries": len(calibration), "mean_labels": label_counts.mean(), "max_labels": label_counts.max(),
    "unique_relevant_articles": len({x for value in calibration.ground_truth for x in value.split()}),
    "exact_calibration_test_overlap": len(set(calibration.query_text.str.casefold()) & set(test.query_text.str.casefold())),
}])
display(label_profile)


## Persist audit artifacts


In [ ]:
summary.to_csv(OUT / "dataset_summary.csv", index=False)
field_stats.to_csv(OUT / "field_statistics.csv", index=False)
html_inventory.to_csv(OUT / "html_tags.csv", index=False)
class_inventory.to_csv(OUT / "html_classes.csv", index=False)
image_manifest.to_parquet(OUT / "image_manifest.parquet", index=False)
audit = {"parsed_articles": str(parsed_path), **image_summary.iloc[0].to_dict(), **label_profile.iloc[0].to_dict()}
save_json(audit, OUT / "audit.json")
print(f"Saved audit artifacts to {OUT}")


## Interpretation

Use `field_statistics.csv` to detect empty/noisy fields and `image_manifest.parquet` as the input to the OCR ablation. Later notebooks must read `parsed_articles.parquet`; they do not reparse HTML.
